In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
df = pd.read_csv('/content/sample_data/Crop_recommendation_missing.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,N,P,K,temperature,humidity,ph,rainfall,label
0,90.0,42.0,43.0,20.879744,82.002744,6.502985,202.935536,rice
1,85.0,58.0,41.0,NaN,80.319644,7.038096,226.655537,rice
2,60.0,55.0,44.0,23.004459,82.320763,7.840207,263.964248,rice
3,74.0,35.0,40.0,26.491096,80.158363,6.980401,242.864034,rice
4,78.0,NaN,42.0,20.130175,81.604873,7.628473,262.717340,rice


##Show Missing Values

In [ ]:
# Calculate the number of missing values for each column
missing_values_count = df.isnull().sum()

# Print the results to identify features needing imputation
print("Missing values in each column:")
print(missing_values_count)

Missing values in each column:
N              109
P              109
K              113
temperature    121
humidity       114
ph              98
rainfall       109
label            0
dtype: int64


## Impute Missing Values by Label Mean


In [ ]:
# List of columns with missing values to be imputed
columns_to_impute = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

# Impute missing values using the mean of the column grouped by 'label'
for col in columns_to_impute:
    df[col] = df[col].fillna(df.groupby('label')[col].transform('mean'))

# Verify if there are any remaining missing values
remaining_nulls = df.isnull().sum()
print("Missing values after imputation:")
print(remaining_nulls)

# Display the first few rows of the cleaned DataFrame
df.head()

Missing values after imputation:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64


,N,P,K,temperature,humidity,ph,rainfall,label
0,90.0,42.000000,43.0,20.879744,82.002744,6.502985,202.935536,rice
1,85.0,58.000000,41.0,23.674081,80.319644,7.038096,226.655537,rice
2,60.0,55.000000,44.0,23.004459,82.320763,7.840207,263.964248,rice
3,74.0,35.000000,40.0,26.491096,80.158363,6.980401,242.864034,rice
4,78.0,47.578947,42.0,20.130175,81.604873,7.628473,262.717340,rice


## Prepare Features and Target


In [ ]:
# Separate features and target
X = df.drop(columns=['label'])
y = df['label']

# Verify the shapes
print(f'Shape of X: {X.shape}')
print(f'Shape of y: {y.shape}')

# Display the first few rows of X to confirm correct columns
X.head()

Shape of X: (2200, 7)
Shape of y: (2200,)


,N,P,K,temperature,humidity,ph,rainfall
0,90.0,42.000000,43.0,20.879744,82.002744,6.502985,202.935536
1,85.0,58.000000,41.0,23.674081,80.319644,7.038096,226.655537
2,60.0,55.000000,44.0,23.004459,82.320763,7.840207,263.964248
3,74.0,35.000000,40.0,26.491096,80.158363,6.980401,242.864034
4,78.0,47.578947,42.0,20.130175,81.604873,7.628473,262.717340


## Split Data into Train and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes to verify the split
print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of y_test: {y_test.shape}')

Shape of X_train: (1760, 7)
Shape of X_test: (440, 7)
Shape of y_train: (1760,)
Shape of y_test: (440,)


## Initialize Base Classifiers

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Initialize base classifiers
# SVC requires probability=True to be used in a soft-voting ensemble later
svc_clf = SVC(probability=True, random_state=42)
knn_clf = KNeighborsClassifier()
dt_clf = DecisionTreeClassifier(random_state=42)

print("Base classifiers (SVC, KNN, Decision Tree) have been initialized.")

Base classifiers (SVC, KNN, Decision Tree) have been initialized.


## Create and Train Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier

# Create the list of base estimators
estimators = [
    ('svc', svc_clf),
    ('knn', knn_clf),
    ('dt', dt_clf)
]

# Initialize the VotingClassifier with soft voting
voting_clf = VotingClassifier(estimators=estimators, voting='soft')

# Train the VotingClassifier
voting_clf.fit(X_train, y_train)

print("Voting Classifier ensemble model has been successfully trained.")

Voting Classifier ensemble model has been successfully trained.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Predict the labels for the test set
y_pred = voting_clf.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# Display the results
print(f'Voting Classifier Accuracy: {accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Voting Classifier Accuracy: 0.9886

Classification Report:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        23
      banana       1.00      1.00      1.00        21
   blackgram       1.00      1.00      1.00        20
    chickpea       1.00      1.00      1.00        26
     coconut       1.00      1.00      1.00        27
      coffee       1.00      1.00      1.00        17
      cotton       1.00      1.00      1.00        17
      grapes       1.00      1.00      1.00        14
        jute       0.88      0.96      0.92        23
 kidneybeans       1.00      1.00      1.00        20
      lentil       0.92      1.00      0.96        11
       maize       1.00      1.00      1.00        21
       mango       1.00      1.00      1.00        19
   mothbeans       1.00      0.96      0.98        24
    mungbean       1.00      1.00      1.00        19
   muskmelon       1.00      1.00      1.00        17
      orange       1.0

## สรุปผลการดำเนินงาน (Final Task)

จากการสร้างโมเดล **Voting Classifier** โดยการรวมพลังของ 3 อัลกอริทึม ได้แก่ **SVC, KNN, และ Decision Tree** พบว่า:

1. **ประสิทธิภาพ**: โมเดลสามารถทำนายประเภทของพืชได้อย่างแม่นยำสูงมาก (Accuracy ประมาณ 98-99%)
2. **ข้อดีของการใช้ Ensemble**: การใช้ Voting แบบ 'soft' ช่วยให้โมเดลตัดสินใจจากความน่าจะเป็นที่คำนวณจากทุกโมเดลย่อย ทำให้ผลลัพธ์มีความเสถียรและลดโอกาสผิดพลาดจากโมเดลใดโมเดลหนึ่งเพียงตัวเดียว
3. **การจัดการข้อมูล**: การเติมค่าว่าง (Imputation) ด้วยค่าเฉลี่ยแยกตามกลุ่ม label มีส่วนสำคัญอย่างมากที่ทำให้โมเดลเรียนรู้ลักษณะเฉพาะของพืชแต่ละชนิดได้อย่างถูกต้องครับ